# 08 — The Quantum Amari Bridge

The single identity the whole quantum arc has been walking toward: the entropic quantum optimal-transport plan is the **Umegaki relative-entropy projection** of the matrix-exponential Gibbs kernel $K = \exp(-C/\varepsilon)$ onto the set of couplings. The operator lift of `03/11`'s classical Amari identity — transport geometry *is* information geometry, now for density matrices. (*Amari's bridge* is the course's own name for that identity, as `03/11` explains; you will not find it under that name in the literature.)

**Prerequisites:** `04/06_entropic_qot_gibbs`, `04/07_operator_sinkhorn_iteration`, and `03/11_amari_bridge`.

**What you'll be able to do**
- State **Amari's quantum bridge**: $\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K) = \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$, so the entropic plan is the Umegaki projection of $K$ onto the coupling set ($K$ is a positive operator, not a state, so this relative entropy can be negative).
- **Verify the identity numerically** on a full-rank commuting pair, reading both sides and seeing them agree to machine precision.
- Read the bridge as the **operator lift** of `03/11`'s classical KL-projection identity — the same Bregman geometry, raised from probability vectors to operators.
- Place the entropic object among the **three QOT objects** the course keeps reconciled, and return to the Sinkhorn/Amari rows of the classical-to-quantum dictionary, whose quantum side you have now built for three of the four.

Welcome back, and take a breath of satisfaction before we begin — you have done the hard building. In `04/06` you wrote the entropic quantum OT problem and met the matrix-exponential Gibbs kernel at its centre; in `04/07` you *ran* the operator Sinkhorn iteration and watched it walk, by alternating rescalings of that kernel, to the very optimum the solver returned on a commuting pair. You have the optimum, and you have an algorithm that reaches it in that commuting setting. What you do not yet have is the answer to a quieter question those two notebooks kept raising: *what is that optimum, geometrically?* Why does rescaling a kernel onto the marginals land exactly there? Today we answer it. The entropic plan turns out to be a **projection** — and the thing it projects is the Gibbs kernel, and the geometry it projects under is the Umegaki relative entropy you first met in `02/10`. This is the operator version of the bridge `03/11` built for probability vectors, and it is one of the most beautiful equations in the course.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.composite import partial_trace, tensor
from qot_course.quantum_ot.sdp import quadratic_position_cost
from qot_course.quantum_ot.sinkhorn import (
    matrix_gibbs_kernel,
    quantum_sinkhorn_sdp,
    umegaki_kl_to_kernel,
)

np.random.seed(42)
viz.use_course_style()

## Amari's quantum bridge — the entropic plan is a Umegaki projection

Recall the move `03/11` made in the classical world. The entropic OT objective and the KL divergence to the Gibbs kernel are, up to an additive constant, the *same function*: plug $K_{ij} = e^{-C_{ij}/\varepsilon}$ into $\varepsilon\,\mathrm{KL}(P\,\|\,K)$ and the $-\log K_{ij} = C_{ij}/\varepsilon$ inside turns it into $\langle C, P\rangle - \varepsilon H(P) + \text{const}$. So minimising the entropic cost over couplings *is* minimising $\mathrm{KL}(P\,\|\,K)$ over couplings — the Sinkhorn plan is the KL projection of the kernel onto the transportation polytope.

The quantum lift keeps the algebra and changes the objects. The Shannon entropy becomes von Neumann's, the entrywise kernel becomes the matrix exponential of `04/06`, and the classical KL becomes the **Umegaki relative entropy** $S_{\mathrm{Umegaki}}(\rho\,\|\,\sigma) = \mathrm{tr}\big(\rho(\log\rho - \log\sigma)\big)$ you met in `02/10`. Substitute $K = \exp(-C/\varepsilon)$, so that $\log K = -C/\varepsilon$, into the operator relative entropy and the same cancellation happens:

$$
\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K)
   \;=\; \varepsilon\,\mathrm{tr}\big(\rho_{AB}\log\rho_{AB}\big) - \varepsilon\,\mathrm{tr}\big(\rho_{AB}\log K\big)
   \;=\; \underbrace{\mathrm{tr}(C\,\rho_{AB})}_{\text{transport cost}} \;-\; \varepsilon\,\underbrace{\big(-\mathrm{tr}(\rho_{AB}\log\rho_{AB})\big)}_{S(\rho_{AB})}.
$$

The right-hand side is exactly the entropic QOT objective of `04/06`. There is no additive constant here, whatever the marginals: $\log K = -C/\varepsilon$ exactly, so the two sides are equal on the nose for every density matrix $\rho_{AB}$. The reading is the punchline:

$$
\boxed{\;\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K) \;=\; \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})\;}
$$

so **the entropic quantum OT plan is the Umegaki relative-entropy projection of the matrix-exponential Gibbs kernel $K$ onto the coupling set.** Minimising the entropic transport objective and projecting $K$ in Umegaki geometry are one and the same. One word of care: $K$ is a positive operator, not a state, since its trace is not $1$. So $S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K)$ can be negative, and Klein's inequality, which makes the relative entropy between two states non-negative, does not apply to it. Normalising $K$ to $K/\mathrm{tr}\,K$ shifts the value by the constant $\log \mathrm{tr}\,K$ and leaves the minimiser unchanged.

We verify the identity numerically. We need the entropic plan to be **full-rank** so that `umegaki_kl_to_kernel`, which takes matrix logarithms of the plan, is on clean numerical ground — and `04/06` already told us which pair gives that. The headline $|+\rangle\langle+|$ versus $I/2$ pair would be a trap: as you proved in `04/04`, a pure marginal leaves only one coupling, the product $|+\rangle\langle+| \otimes I/2$, which has rank two and sits on the boundary of the PSD cone, where $\log\rho_{AB}$ is not defined. So we keep the commuting pair from `04/06`–`04/07` — the quadratic position cost on positions $\{0, 1, 2\}$ with $\rho_A = \mathrm{diag}(0.5, 0.3, 0.2)$ and $\rho_B = \mathrm{diag}(0.2, 0.3, 0.5)$ — whose entropic plan is strictly positive definite. We compute both sides at $\varepsilon = 0.5$ and read them off.

In [ ]:
# The full-rank commuting pair from 04/06-07: quadratic position cost on positions
# {0, 1, 2}, two diagonal states with room to spread. Its entropic plan is strictly
# positive definite, so the matrix logs inside umegaki_kl_to_kernel are clean -- unlike
# the |+><+| vs I/2 pair, whose only coupling (04/04) sits on the PSD boundary.
C = quadratic_position_cost([0.0, 1.0, 2.0])
rho_a = np.diag([0.5, 0.3, 0.2]).astype(complex)
rho_b = np.diag([0.2, 0.3, 0.5]).astype(complex)

epsilon = 0.5

# Solve the entropic QOT SDP (04/06) for the plan, and build the matrix Gibbs kernel.
# (A CLARABEL "solution may be inaccurate" warning may print -- it is the interior-point
# method reporting it stopped near, not exactly at, its tightened tolerance. We keep it.)
_, plan = quantum_sinkhorn_sdp(rho_a, rho_b, C, epsilon=epsilon)
K = matrix_gibbs_kernel(C, epsilon)

# The plan must be full-rank for the Umegaki logarithm to be well-conditioned.
plan_eigs = np.linalg.eigvalsh(0.5 * (plan + plan.conj().T))
print(f"entropic plan: {int(np.sum(plan_eigs > 1e-10))}/{plan.shape[0]} eigenvalues above 1e-10 "
      f"(smallest {plan_eigs.min():.2e}) -- full rank, logs are clean")
print()

# LHS of Amari's quantum bridge:  eps * S_Umegaki(P || K) = eps * tr[P (log P - log K)].
lhs = epsilon * umegaki_kl_to_kernel(plan, K)

# RHS:  tr(C P) - eps * S(P), the entropic objective of 04/06, assembled from its pieces.
transport = float(np.real(np.trace(C @ plan)))
vals = plan_eigs[plan_eigs > 1e-12]
S_plan = float(-np.sum(vals * np.log(vals)))  # von Neumann entropy, natural-log convention
rhs = transport - epsilon * S_plan

print(f"LHS   eps * S_Umegaki(P || K)        = {lhs:.6f}")
print(f"RHS   tr(C P) - eps * S(P)           = {rhs:.6f}")
print(f"        tr(C P)                      = {transport:.6f}")
print(f"        S(P)                         = {S_plan:.6f} nats  ({S_plan / np.log(2):.6f} bits)")
print()
print(f"|LHS - RHS|                          = {abs(lhs - rhs):.2e}")
print(f"Amari's quantum bridge holds?          {bool(np.isclose(lhs, rhs, atol=1e-6))}")

**Read the output.** The plan is full-rank — all nine eigenvalues are positive, the smallest at about $1.2 \times 10^{-7}$ — so the matrix logarithm inside the Umegaki relative entropy is defined, as we arranged, with that one small eigenvalue as its most delicate input. Then the two sides of the boxed identity print the **same number** (here a small negative value, since at $\varepsilon = 0.5$ the entropy bonus $\varepsilon\,S(P)$ slightly outweighs the transport cost), and their difference is at the $10^{-16}$ floor, machine precision. That agreement owes nothing to the solver: the identity is algebra, and it holds for *any* positive definite matrix put in place of the plan. There is no fitted constant and no approximation: $\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K)$ and $\mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$ are *literally the same quantity*, evaluated two ways.

That equality is the whole point. Because the two objectives coincide, the coupling that minimises the entropic transport cost is the very same coupling that minimises the Umegaki relative entropy to $K$ over the coupling set — it is the **Umegaki projection of the Gibbs kernel**. In the **commuting case**, where the states and the cost share an eigenbasis, the operator Sinkhorn iteration you ran in `04/07` computes exactly this projection: each alternating rescaling is then a relative-entropy projection onto one marginal constraint, as in the alternating KL projections behind classical Sinkhorn.

Outside the commuting case the two can part ways, and it is worth seeing why. The Umegaki projection onto one marginal constraint has the form $\exp(\log K + U \otimes I)$, while a rescaling step of `04/07` produces $(X \otimes I)\,K\,(X \otimes I)$. These agree when $K$ commutes with the factors; otherwise they can differ. On the non-commuting qubit example at the end of `04/07` they do: there the fixed point of the iteration is a different coupling from the Umegaki projection, with a higher entropic objective. They need not differ: on that same pair, a cost of the form $C_A \otimes I + I \otimes C_B$ has the product state as its optimum, and the iteration finds it. The projection is what the entropic SDP computes; the iteration of `04/07` is the operator-scaling solution in its own right, as that notebook's caveat says.

### Is the plan really the minimiser?

The identity tells you that the two objectives are one function. It does not tell you that the solver found that function's minimum. Here is a direct test. The coupling set is convex, so for any other coupling $Q$ every point $(1-t)\,P^\star + t\,Q$ is a coupling too. If $P^\star$ is the Umegaki projection of $K$, then $\varepsilon\,S_{\mathrm{Umegaki}}(\cdot\,\|\,K)$ must rise as you walk away from it along every such segment. We walk toward the product $\rho_A \otimes \rho_B$ and toward 19 random quantum couplings, built by adding to the product a Hermitian perturbation whose two partial traces vanish.

In [ ]:
# Minimality test: walk from the entropic plan P* toward other couplings Q.
# The coupling set is convex, so every point (1 - t) P* + t Q on the segment is a coupling.
# If P* is the Umegaki projection, eps * S_Umegaki(. || K) can only rise as t leaves 0.
f_opt = epsilon * umegaki_kl_to_kernel(plan, K)
product = tensor(rho_a, rho_b)

def zero_marginal_direction(D):
    """Hermitian part of D with both partial traces removed (dims 3 x 3)."""
    D = 0.5 * (D + D.conj().T)
    tr_b = partial_trace(D, keep=[0], dims=[3, 3])
    tr_a = partial_trace(D, keep=[1], dims=[3, 3])
    eye = np.eye(3)
    return D - np.kron(tr_b, eye) / 3 - np.kron(eye, tr_a) / 3 + np.trace(D) * np.eye(9) / 9

# Other couplings: the product, and 19 random quantum couplings (product + a zero-marginal
# Hermitian perturbation, scaled so the result stays positive definite).
rng_min = np.random.default_rng(8)
lam_min = np.linalg.eigvalsh(product).min()
couplings = [product]
for _ in range(19):
    H = zero_marginal_direction(rng_min.normal(size=(9, 9)) + 1j * rng_min.normal(size=(9, 9)))
    couplings.append(product + 0.9 * lam_min / np.abs(np.linalg.eigvalsh(H)).max() * H)

ts = np.linspace(0.0, 1.0, 21)
gaps = np.array([[epsilon * umegaki_kl_to_kernel((1 - t) * plan + t * Q, K) - f_opt for t in ts]
                 for Q in couplings])

print(f"largest marginal error among the 20 couplings     = "
      f"{max(np.linalg.norm(partial_trace(Q, keep=[0], dims=[3, 3]) - rho_a) for Q in couplings):.1e}")
print(f"smallest gap above the optimum (t >= 0.05)        = {gaps[:, 1:].min():.4f}")
print(f"gap at the product state (t = 1)                  = {gaps[0, -1]:.4f}")
print(f"every curve rises monotonically from t = 0?         {bool(np.all(np.diff(gaps, axis=1) > 0))}")

fig, ax = plt.subplots(figsize=(8, 5))
for g in gaps[1:]:
    ax.plot(ts, g, color=COLORS["quantum"], alpha=0.45, linewidth=1.2)
ax.plot([], [], color=COLORS["quantum"], alpha=0.45, label="toward 19 random quantum couplings")
ax.plot(ts, gaps[0], color=COLORS["highlight"], linewidth=2.5, label=r"toward the product $\rho_A \otimes \rho_B$")
ax.axhline(0.0, color=COLORS["muted"], linestyle="--", linewidth=1.0)
ax.set_xlabel(r"position $t$ on the segment $(1-t)\,P^\star + t\,Q$")
ax.set_ylabel(r"$\varepsilon\,S_{\mathrm{Umegaki}}(\cdot\,\|\,K)$ minus its value at $P^\star$")
ax.set_title("Every walk away from the entropic plan costs relative entropy")
ax.legend()
plt.show()

**Read the figure.** Every curve starts at zero, at the entropic plan, and climbs. The smallest rise anywhere past $t = 0.05$ is $0.0119$, and walking all the way to the product costs $0.6720$. No direction goes below the dashed line, so none of these 20 couplings, nor any point between them and $P^\star$, beats the plan the solver returned. The random couplings are small perturbations of the product, which is why their curves bunch around the highlighted one. This is a check on this example, not a proof: twenty directions out of a continuum. The proof is the convexity of the Umegaki relative entropy, which makes a point with no descent direction the global minimiser.

## Bringing it together — the operator lift of `03/11`

Step back and see what has closed. In `03/11` you proved that the classical entropic OT plan is the **KL projection** of the entrywise Gibbs kernel onto the transportation polytope, and read it as the meeting of two geometries: the *information* geometry of Module 2 (the KL divergence) and the *transport* geometry of Module 3 (couplings) turned out to be a single Bregman geometry seen from two sides. The identity you verified above is that same statement, lifted **word for word** to operators:

- the probability vector $P$ became the density matrix $\rho_{AB}$,
- the Shannon entropy $H$ became the von Neumann entropy $S$,
- the entrywise kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ became the matrix exponential $K = \exp(-C/\varepsilon)$,
- and the classical KL divergence became the **Umegaki relative entropy** of `02/10`.

The same Bregman picture survives the lift intact: the quantum information geometry of Module 2 and the quantum transport geometry of this module are one geometry, raised from probability vectors to density matrices. This is why the row that closed the dictionary update of `03/11` — *entropic plan = KL projection* $\leftrightarrow$ *entropic quantum plan = Umegaki projection* — is no longer a promise about the future. You have now made both halves true, in code. One word of care about the algorithm: classical Sinkhorn *computes* the KL projection, while the operator Sinkhorn iteration is sure to compute the Umegaki projection in the commuting case and can miss it outside, as you saw above.

This entropic plan is also one of the **three QOT objects** the course commits to keeping reconciled. You have met all three: the **Bures–Wasserstein bridge** (the closed-form $W_2$ between Gaussian/qubit states, the literal doorway into the quantum world), the **coupling SDP** (the operator Kantorovich problem of `04/05`, minimising $\mathrm{tr}(C\,\rho_{AB})$ over quantum couplings), and the **entropic object** you have now given a geometric identity. They are three faces of quantum optimal transport, and the natural next question is when they *agree* — when the entropic plan limits to the SDP optimum, when the SDP value matches the Bures formula. That reconciliation, with the regimes laid out side by side, is exactly the work of `04/09`.

## The QOT dictionary rows

The classical-to-quantum dictionary has grown all module long, one notebook at a time. The entropic arc — `04/06`, `04/07`, and this notebook — returns to the four Sinkhorn/Amari rows, each one the operator lift of a classical entry from the Sinkhorn arc of Module 3 (`03/08`–`03/11`). They first entered the dictionary in `03/11`, with their quantum side still ahead of you, and this arc earns them a second time: you have now built the quantum side of three of them. The quantum potentials, the dual side of the SDP, are named and not built in this course, as their row says.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| Kantorovich potentials $\varphi, \psi$ (LP duals) | quantum potentials (SDP duals; not built in this course) |
| Gibbs kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ (entrywise) | matrix-exponential kernel $K = \exp(-C/\varepsilon)$ |
| Sinkhorn iteration (row/column matrix scaling) | quantum Sinkhorn (operator scaling: $(X\otimes Y)\,K\,(X\otimes Y)$, partial-trace fixed point) |
| entropic plan = KL projection of $K = e^{-C/\varepsilon}$ (computed by Sinkhorn) | entropic quantum plan = Umegaki projection of $K = \exp(-C/\varepsilon)$ (reached by operator Sinkhorn in the commuting case) |

Read down the right column and you have the entire entropic story of this module in four lines: the dual variables became operators, a step this course names without taking; the kernel became a matrix exponential; the alternating scaling became operator scaling; and the projection that explains it all swapped the classical KL for the Umegaki relative entropy. Every row follows the same principle — *swap probability vectors for density matrices, swap Shannon/KL for von Neumann/Umegaki* — and the last row carries its limit with it: the projection identity holds for every pair of states, while the operator Sinkhorn iteration is sure to reach that projection in the commuting case and can miss it outside.

## Your turn

**Warm-up.** Re-run the identity check at a second regularisation strength — say $\varepsilon = 1.0$ — on the same commuting pair. Solve with `quantum_sinkhorn_sdp`, rebuild `K = matrix_gibbs_kernel(C, epsilon)`, and confirm that `epsilon * umegaki_kl_to_kernel(plan, K)` still equals `tr(C P) - epsilon * S(P)` to machine precision. Which of the two terms on the right grows as you raise $\varepsilon$, and does the sign of the total match what you saw at $\varepsilon = 0.5$?

**Go further.** The boxed identity holds for *any* coupling, not only the optimum — it is an algebraic rewrite, not an optimality condition. Take the **product state** $\rho_A \otimes \rho_B$ (build it with `tensor(rho_a, rho_b)` from `qot_course.quantum.composite`), which is a perfectly valid coupling but not the entropic minimiser, and check that the two sides of $\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K) = \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$ still agree on it. Then compare the value $\varepsilon\,S_{\mathrm{Umegaki}}$ at the product to its value at the entropic optimum from this section — which is smaller, and why does the projection picture predict that ordering?

**Challenge.** Connect the identity to the algorithm you ran in `04/07`. Work in the commuting case first. Argue, in words, why "the entropic plan is the Umegaki projection of $K$ onto the coupling set" makes the operator Sinkhorn iteration an *iterative Bregman projection* there: what does each half-step (rescaling so that one partial trace equals its target marginal) do to the Umegaki relative entropy, and why does alternating two such projections converge to the joint minimiser? You met the classical version of this argument in `03/11`; restate it for the Umegaki geometry, naming what plays the role of the affine marginal constraints. Then find the step of your argument that needs commutation: the projection onto one marginal constraint is $\exp(\log K + U \otimes I)$, the rescaling is $(X \otimes I)\,K\,(X \otimes I)$ — when are these the same operator?

## What you built

- You stated and **verified Amari's quantum bridge** — $\varepsilon\,S_{\mathrm{Umegaki}}(\rho_{AB}\,\|\,K) = \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$ — on a full-rank commuting pair, watching both sides agree to machine precision, so you know the entropic QOT plan **is** the Umegaki relative-entropy projection of the matrix-exponential Gibbs kernel onto the coupling set.
- You read the bridge as the **operator lift of `03/11`**: the same Bregman geometry that married information and transport for probability vectors survives intact for density matrices, with the Umegaki relative entropy in the seat the classical KL held.
- You placed the entropic plan among the **three QOT objects** — Bures–Wasserstein bridge, coupling SDP, entropic — and saw why their reconciliation is the natural next question.
- You returned to the four Sinkhorn/Amari rows of the classical-to-quantum **dictionary** and built the quantum side of three of them: the Gibbs kernel, the Sinkhorn iteration, and the Amari projection itself, each an operator lift of its Module-3 twin. The remaining row, the quantum potentials, is named and not built in this course.

Sit for a moment with what this identity says. The information geometry you built in Module 2 and the transport geometry you built across Modules 3 and 4 were never two subjects — they are one geometry, and you have now seen it from both the classical and the quantum side. The entropic optimum you computed by solver, and reached by hand-run iteration on a commuting pair, has a name: it is a projection, in the relative-entropy geometry, of a kernel you can write down in closed form. That is as deep as this course goes, and you got here by building every piece yourself.

## What's next

In `09_the_qot_zoo` we put the three QOT objects on one bench and ask the reconciliation question head-on: when does the entropic plan coincide with the coupling-SDP optimum, when does the SDP value match the Bures–Wasserstein formula, and what regime sends them apart? Today's identity is the entropic object's entry in that comparison; `04/09` is where all three meet.

## References

- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016), sec. 7.5.
- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *Advances in Neural Information Processing Systems* **26** (2013). DOI:10.48550/arXiv.1306.0895
- G. Peyré, L. Chizat, F.-X. Vialard & J. Solomon, "Quantum entropic regularization of matrix-valued optimal transport", *European Journal of Applied Mathematics* **30**, 1079–1102 (2019). DOI:10.1017/S0956792517000274
- H. Umegaki, "Conditional expectation in an operator algebra. IV. Entropy and information", *Kodai Mathematical Seminar Reports* **14**, 59–85 (1962). DOI:10.2996/kmj/1138844604
- D. Feliciangeli, A. Gerolin & L. Portinale, "A non-commutative entropic optimal transport approach to quantum composite systems at positive temperature", *Journal of Functional Analysis* **285**, 109963 (2023). DOI:10.1016/j.jfa.2023.109963 — entropic optimal transport on density matrices, with the von Neumann entropy as regulariser.
- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5

**Previous:** `notebooks/04_QuantumOptimalTransport/07_operator_sinkhorn_iteration.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/09_the_qot_zoo.ipynb`